[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C36_GPU_Kernels_Course/01_execution_model/01_execution_model.ipynb)

# 01 · GPU 执行模型（用 numpy 模拟）

目标：把 **thread / warp / block / grid** 四级层级、**SIMT 分支发散**、**occupancy 木桶计算**、**延迟隐藏** 用 numpy 模拟出来，并用 `assert` 验证。

路线：线程→数据映射(vector add) → warp 分组 → 分支发散代价 → 占用率计算 → 延迟隐藏直觉 → grid-stride → ✏️ 练习 → 📖 答案 → 🧪 真实 GPU 规格胶囊。

> 心智模型：**一个 block = 一段循环；一个 warp = 长度 32 的切片；shared/寄存器 = 临时数组**。我们写的是并行*结构*，不是性能。

## 1 · 线程→数据映射：模拟 vector add 的网格

GPU 不写循环，而是为每个 `idx` 开一个线程并行执行 `c[idx]=a[idx]+b[idx]`。
全局下标公式：`idx = block_id * block_size + thread_id`。

我们用循环**模拟**这个并行：枚举每个 (block_id, thread_id)，正是 GPU 实际并行做的事。注意 **边界保护** `if idx < n`。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def vadd_grid_sim(a, b, block_size=256):
    '''模拟 GPU 启动 grid=ceil(n/block_size) 个 block 来做 a+b。
       用双重循环枚举 (block_id, thread_id)，即硬件并行执行的全部线程。'''
    n = a.shape[0]
    grid = (n + block_size - 1) // block_size      # ceil(n/block_size)
    c = np.empty(n, dtype=a.dtype)
    launched = 0
    for block_id in range(grid):
        for thread_id in range(block_size):
            idx = block_id * block_size + thread_id
            launched += 1
            if idx < n:                            # 边界保护！
                c[idx] = a[idx] + b[idx]
    return c, grid, launched

n = 1000
a = rng.standard_normal(n); b = rng.standard_normal(n)
c, grid, launched = vadd_grid_sim(a, b, block_size=256)
print(f'n={n}, block_size=256 -> grid={grid} blocks, 启动 {launched} 个线程')
print(f'多启动的(越界)线程数 = {launched - n}  <- 它们被 if idx<n 跳过')
assert np.allclose(c, a + b), '对拍 numpy a+b'
assert launched == grid * 256 and grid == 4
print('✅ 对拍 a+b 通过；grid 向上取整产生了多余线程，边界保护救了场')

## 2 · warp：硬件真正的调度粒度（32 线程一组）

block 内的线程不是逐个调度的，而是按 **warp（32 个）** 锁步执行。一个 block 被切成 `block_size/32` 个 warp。
下面把一个 block 的线程按 warp 分组，这对理解后面的分支发散至关重要。

In [ ]:
WARP = 32

def warps_of_block(block_size):
    '''返回每个 warp 包含的线程局部下标。'''
    n_warps = (block_size + WARP - 1) // WARP
    return [list(range(w * WARP, min((w + 1) * WARP, block_size))) for w in range(n_warps)]

for bs in [32, 128, 256]:
    ws = warps_of_block(bs)
    print(f'block_size={bs:3d} -> {len(ws)} 个 warp，每个 {len(ws[0])} 线程')
assert len(warps_of_block(256)) == 8
assert len(warps_of_block(100)) == 4 and len(warps_of_block(100)[-1]) == 4  # 不满一个 warp
print('✅ warp 分组正确；注意 block_size=100 的最后一个 warp 只有 4 个有效线程（其余空转）')

## 3 · 分支发散：SIMT 的代价

一个 warp 锁步执行同一条指令。若 warp 内线程走了 `if/else` 的不同分支，硬件只能**串行**执行每条被走到的分支路径（不参与的线程空转）。

**代价 ≈ 该 warp 走到的不同分支条数**。我们模拟：给每个线程一个分支决策，统计每个 warp 的「发散惩罚」。

In [ ]:
def divergence_cost(branch_ids, block_size=256):
    '''branch_ids[i] = 线程 i 走的分支编号(如 0/1)。
       返回 (总代价, 无发散时的理想代价)。代价单位=warp 需串行执行的分支路径数。'''
    warps = warps_of_block(block_size)
    total = 0
    for w in warps:
        distinct = len(set(branch_ids[i] for i in w))   # 这个 warp 走了几条不同的路
        total += distinct                               # 每条路串行执行一次
    ideal = len(warps)                                  # 完全不发散：每 warp 只 1 条路
    return total, ideal

bs = 256
# 情形 A：按 warp 对齐分块——同一 warp 内全走同一支（无发散）
branch_aligned = [(i // WARP) % 2 for i in range(bs)]
# 情形 B：按奇偶分支——每个 warp 内一半走 0、一半走 1（处处发散）
branch_interleaved = [i % 2 for i in range(bs)]

ca, ideal = divergence_cost(branch_aligned, bs)
cb, _     = divergence_cost(branch_interleaved, bs)
print(f'理想(无发散)代价      = {ideal}')
print(f'A 按warp对齐  代价   = {ca}  (减速 {ca/ideal:.1f}x)')
print(f'B 奇偶交错    代价   = {cb}  (减速 {cb/ideal:.1f}x)')
assert ca == ideal, 'warp 对齐应当无发散'
assert cb == 2 * ideal, '奇偶交错应当每 warp 走 2 条路 -> 2x'
print('✅ 同样的分支比例，布局不同导致 2x 差距 —— 让同 warp 线程走同一支！')

## 4 · Occupancy：三种资源的木桶效应

占用率 = 实际驻留 warp 数 ÷ SM 最大 warp 数。受三者**最小值**封顶：寄存器、shared memory、block/线程硬上限。

对每种资源算「最多能容纳几个 block」，取最小，换算成 warp 占比。下面用接近真实 GPU 的硬限做这个木桶计算。

In [ ]:
def occupancy(block_size, regs_per_thread, smem_per_block,
              max_warps_per_sm=64, max_blocks_per_sm=32,
              regs_per_sm=65536, smem_per_sm=49152):
    '''返回 (占用率, 受限资源名)。数字量级取自 Ampere 级 SM。'''
    warps_per_block = (block_size + WARP - 1) // WARP
    # 各资源分别允许多少个 block 同时驻留
    by_threads = max_warps_per_sm // warps_per_block
    by_blocks  = max_blocks_per_sm
    by_regs    = regs_per_sm // (regs_per_thread * block_size) if regs_per_thread > 0 else 10**9
    by_smem    = smem_per_sm // smem_per_block if smem_per_block > 0 else 10**9
    limits = {'threads/warps': by_threads, 'blocks': by_blocks, 'registers': by_regs, 'shared_mem': by_smem}
    active_blocks = min(limits.values())
    binding = min(limits, key=limits.get)
    active_warps = active_blocks * warps_per_block
    occ = active_warps / max_warps_per_sm
    return occ, binding, active_blocks

# 例1：轻量内核(寄存器少、不用shared) -> 满占用
occ1, bind1, ab1 = occupancy(256, regs_per_thread=32, smem_per_block=0)
print(f'轻量内核 : occupancy={occ1:.0%}  受限于 {bind1} ({ab1} blocks/SM)')
# 例2：寄存器大户 -> 占用率被寄存器拖垮
occ2, bind2, ab2 = occupancy(256, regs_per_thread=128, smem_per_block=0)
print(f'寄存器大户: occupancy={occ2:.0%}  受限于 {bind2} ({ab2} blocks/SM)')
assert occ1 == 1.0 and bind1 in ('threads/warps', 'blocks')
assert occ2 < occ1 and bind2 == 'registers'
print('✅ 寄存器用量翻 4 倍把占用率从 100% 砍下来 —— 寄存器是最常见的占用率杀手')

## 5 · 延迟隐藏：为什么需要很多 warp

单次访存要 ~数百周期。GPU 靠「谁卡住就切到别的 warp」把延迟藏住。

玩具模型：每个 warp 算 1 周期、然后等内存 `L` 周期。调度器每周期挑一个**就绪**的 warp 执行。我们模拟总耗时，看 warp 数从少到多时，内存延迟如何被**摊销**到几乎消失。

In [ ]:
def simulate_latency_hiding(n_warps, mem_latency=8, compute=1, total_compute_ops=64):
    '''极简轮转调度：每个 warp 需要做 total_compute_ops 次「算1周期+等mem_latency周期」。
       调度器每周期执行一个就绪 warp 的 1 周期计算。返回完成所有工作的总周期数。'''
    ready_at = [0] * n_warps          # 每个 warp 下次就绪的时刻
    remaining = [total_compute_ops] * n_warps
    t = 0
    busy_cycles = 0                   # 算术单元真正干活的周期
    while any(r > 0 for r in remaining):
        # 找一个已就绪且还有活的 warp
        cand = [w for w in range(n_warps) if remaining[w] > 0 and ready_at[w] <= t]
        if cand:
            w = cand[0]
            busy_cycles += 1
            remaining[w] -= 1
            ready_at[w] = t + 1 + mem_latency   # 算1周期后，要等mem_latency才再就绪
        t += 1
    return t, busy_cycles

print(f"{'warp数':>6s} {'总周期':>8s} {'利用率':>8s}")
for nw in [1, 2, 4, 9, 16]:
    cycles, busy = simulate_latency_hiding(nw, mem_latency=8)
    util = busy / cycles
    print(f'{nw:>6d} {cycles:>8d} {util:>7.0%}')
# 1 个 warp 时利用率 ~1/(1+8)=11%；warp 数 >= 1+mem_latency 时利用率->100%
c1, b1 = simulate_latency_hiding(1, mem_latency=8)
c9, b9 = simulate_latency_hiding(9, mem_latency=8)
assert b1 / c1 < 0.2, '单 warp 利用率应很低'
assert b9 / c9 > 0.95, 'warp 数到 1+latency 时应几乎打满'
print('\n✅ 关键：要藏住 L 周期延迟，约需 1+L 个在飞 warp。这就是占用率的意义。')

## 6 · grid-stride loop：让线程数与数据量解耦

与其为每个元素开一个线程，不如开**适量**线程，每个线程跨步处理多个元素：`for i in range(idx, n, grid_size)`。
好处：线程数可固定为「刚好填满 SM」，既保占用率又控制启动/调度开销。

In [ ]:
def vadd_grid_stride_sim(a, b, n_blocks=4, block_size=64):
    n = a.shape[0]
    c = np.empty(n, dtype=a.dtype)
    grid_size = n_blocks * block_size       # 总线程数(可远小于 n)
    for block_id in range(n_blocks):
        for thread_id in range(block_size):
            idx = block_id * block_size + thread_id
            i = idx
            while i < n:                    # grid-stride：每个线程跨步多吃几口
                c[i] = a[i] + b[i]
                i += grid_size
    return c, grid_size

n = 1000
a = rng.standard_normal(n); b = rng.standard_normal(n)
c, gs = vadd_grid_stride_sim(a, b, n_blocks=4, block_size=64)
print(f'n={n}, 只用 {gs} 个线程(grid-stride)，每线程平均处理 {n/gs:.1f} 个元素')
assert np.allclose(c, a + b)
print('✅ 256 个线程吃下 1000 个元素，结果与 a+b 一致 —— 线程数与数据量解耦')

---
## ✏️ 练习 1：2D 线程映射

把一个 `(H, W)` 的矩阵每个元素映射到一个线程。CUDA 用 2D block/grid，全局坐标：
`row = blockIdx.y*blockDim.y + threadIdx.y`，`col = blockIdx.x*blockDim.x + threadIdx.x`。

实现 `add_2d_sim(A, B, bs_y, bs_x)`：模拟 2D 网格做 `A+B`，含边界保护，返回结果矩阵。

In [ ]:
def add_2d_sim(A, B, bs_y=16, bs_x=16):
    H, W = A.shape
    C = np.zeros_like(A)
    # TODO: 用 grid_y=ceil(H/bs_y), grid_x=ceil(W/bs_x)，四重循环枚举
    #       (block_y, block_x, thread_y, thread_x)，算出 (row,col)，
    #       带边界保护 row<H and col<W，写 C[row,col]=A[row,col]+B[row,col]
    raise NotImplementedError
    return C

In [ ]:
# —— 练习 1 自测 ——
H, W = 37, 50                      # 故意不是 block 整数倍
A = rng.standard_normal((H, W)); B = rng.standard_normal((H, W))
C = add_2d_sim(A, B, bs_y=16, bs_x=16)
assert C.shape == (H, W)
assert np.allclose(C, A + B), '2D 映射结果应等于 A+B（含边界处理）'
print('✅ 练习 1 通过：2D 线程映射 + 边界保护正确')

## ✏️ 练习 2：发散惩罚最小化

给定每个线程的「类别」`labels`（如按数据值分成 0/1/2/3），block_size=256。

(a) 实现 `cost_for_layout(labels)` 复用第 3 节逻辑算发散代价；
(b) 实现 `sorted_layout(labels)`：把线程**按类别排序**后再分 warp，使同 warp 尽量同类，返回排序后代价。
验证：排序后代价 ≤ 原始代价。

In [ ]:
def cost_for_layout(labels, block_size=256):
    # TODO: 复用 divergence_cost，返回总代价(int)
    raise NotImplementedError

def sorted_layout_cost(labels, block_size=256):
    # TODO: 把 labels 排序后再算发散代价（模拟「按类别重排线程」）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
labels = list(rng.integers(0, 4, size=256))
raw = cost_for_layout(labels)
srt = sorted_layout_cost(labels)
ideal = 256 // 32                 # = 8，每 warp 1 条路
assert srt <= raw, '排序后发散不应更差'
assert srt <= raw and ideal <= srt, '排序后趋近理想下界'
print(f'原始代价={raw}  排序后={srt}  理想下界={ideal}')
print('✅ 练习 2 通过：按类别重排线程显著减少 warp 发散')

## ✏️ 练习 3：占用率调参

你的内核每线程用 `R` 个寄存器、每 block 用 `S` 字节 shared memory，block_size=128。

实现 `best_blocksize(regs_per_thread, smem_per_block, candidates)`：在候选 block 尺寸里选**占用率最高**的，返回 `(最佳block_size, 占用率)`。复用第 4 节的 `occupancy`。

In [ ]:
def best_blocksize(regs_per_thread, smem_per_block, candidates=(64,128,256,512,1024)):
    # TODO: 对每个候选算 occupancy(...)[0]，返回占用率最高者 (block_size, occ)
    #       并列时取较小的 block_size
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
bs, occ = best_blocksize(regs_per_thread=64, smem_per_block=0)
assert isinstance(bs, int) and 0 < occ <= 1.0
# 寄存器很重时，更小的 block 往往能拿到更高(或相等)占用率
bs_heavy, occ_heavy = best_blocksize(regs_per_thread=255, smem_per_block=0)
assert occ_heavy <= occ, '寄存器越重，可达占用率不会更高'
# 逐一核对：返回的确实是候选中的最优
all_occ = [occupancy(b, 64, 0)[0] for b in (64,128,256,512,1024)]
assert abs(occ - max(all_occ)) < 1e-9
print(f'最佳 block_size={bs}, occupancy={occ:.0%}')
print('✅ 练习 3 通过：能据资源用量选出占用率最优的 block 尺寸')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def add_2d_sim(A, B, bs_y=16, bs_x=16):
    H, W = A.shape
    C = np.zeros_like(A)
    grid_y = (H + bs_y - 1) // bs_y
    grid_x = (W + bs_x - 1) // bs_x
    for by in range(grid_y):
        for bx in range(grid_x):
            for ty in range(bs_y):
                for tx in range(bs_x):
                    row = by * bs_y + ty
                    col = bx * bs_x + tx
                    if row < H and col < W:
                        C[row, col] = A[row, col] + B[row, col]
    return C

In [ ]:
# 练习 2 参考答案
def cost_for_layout(labels, block_size=256):
    return divergence_cost(labels, block_size)[0]

def sorted_layout_cost(labels, block_size=256):
    return divergence_cost(sorted(labels), block_size)[0]

In [ ]:
# 练习 3 参考答案
def best_blocksize(regs_per_thread, smem_per_block, candidates=(64,128,256,512,1024)):
    best = None
    for b in candidates:
        occ = occupancy(b, regs_per_thread, smem_per_block)[0]
        if best is None or occ > best[1] + 1e-12:
            best = (b, occ)
    return best

---
## 🧪 真实数据胶囊：用真实 GPU 规格算占用率上限

下面是几代 NVIDIA GPU 每个 SM 的**真实**资源上限（公开白皮书）。用它们算「一个内核最多能驻留多少 warp」，体会不同架构的执行配置差异。

In [ ]:
# 各架构每 SM 的真实资源上限（约数，来自架构白皮书）
SM_SPECS = {
    'V100 (Volta)':  dict(max_warps=64, max_blocks=32, regs=65536, smem=98304),
    'A100 (Ampere)': dict(max_warps=64, max_blocks=32, regs=65536, smem=167936),
    'H100 (Hopper)': dict(max_warps=64, max_blocks=32, regs=65536, smem=233472),
}

def occ_on(spec, block_size, regs_per_thread, smem_per_block):
    return occupancy(block_size, regs_per_thread, smem_per_block,
                     max_warps_per_sm=spec['max_warps'], max_blocks_per_sm=spec['max_blocks'],
                     regs_per_sm=spec['regs'], smem_per_sm=spec['smem'])

# 一个用 96 寄存器/线程、48KB shared/block、block=256 的内核
for name, spec in SM_SPECS.items():
    occ, bind, ab = occ_on(spec, 256, regs_per_thread=96, smem_per_block=48*1024)
    print(f'{name:16s} occupancy={occ:>4.0%}  受限于 {bind:13s} ({ab} blocks/SM)')
print('\n观察：同一内核在 shared memory 更大的新架构上，占用率可能从被 shared 限制转为被寄存器/warp 限制。')

**🧪 胶囊练习**：实现 `max_smem_per_block(spec, target_blocks)`：在给定架构上，要让每个 SM 至少驻留 `target_blocks` 个 block，每个 block 最多能用多少 shared memory？（只考虑 shared 这一项约束。）

In [ ]:
def max_smem_per_block(spec, target_blocks):
    # TODO: 返回 floor(spec['smem'] / target_blocks)
    raise NotImplementedError

In [ ]:
# 自测
a100 = SM_SPECS['A100 (Ampere)']
smem = max_smem_per_block(a100, target_blocks=4)
assert smem == a100['smem'] // 4
# 用这个上限，shared 不会成为 4-block 驻留的瓶颈
occ, bind, ab = occ_on(a100, 256, regs_per_thread=32, smem_per_block=smem)
assert ab >= 4, '至少应能驻留 target_blocks 个 block'
print(f'A100 上要驻留 4 个 block，每 block 最多用 {smem} 字节 shared ({smem//1024} KB)')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def max_smem_per_block(spec, target_blocks):
    return spec['smem'] // target_blocks

---
## 🔧 旁注：对应的 Triton 内核长什么样

本课模拟的「线程→数据映射 + 边界保护」，在 Triton 里就是一个真实可跑的 vector-add 内核（伪代码，**本环境不跑**）：

```python
import triton, triton.language as tl

@triton.jit
def vadd_kernel(a_ptr, b_ptr, c_ptr, n, BLOCK: tl.constexpr):
    pid   = tl.program_id(0)                 # 相当于 blockIdx.x
    offs  = pid * BLOCK + tl.arange(0, BLOCK)  # 这一块负责的全局下标
    mask  = offs < n                         # 边界保护 == 我们的 if idx<n
    a = tl.load(a_ptr + offs, mask=mask)     # 合并访问由编译器保证
    b = tl.load(b_ptr + offs, mask=mask)
    tl.store(c_ptr + offs, a + b, mask=mask)

# 启动：grid = (triton.cdiv(n, BLOCK),)  <- 就是我们的 ceil(n/block_size)
```

对应关系：`program_id`↔block_id、`tl.arange`↔block 内线程、`mask`↔边界保护。Triton 把「warp 怎么分、访问怎么合并」自动处理掉——你只需写对**块级**逻辑，正是本课练的东西。

### 小结
- GPU = 海量线程 + 延迟隐藏：靠「谁卡住切谁」用并行藏住数百周期访存延迟。
- 四级层级：thread→**warp(32, 锁步)**→block(shared+同步)→grid；warp 是真实调度粒度。
- 分支发散：同 warp 走不同分支会串行化，代价≈分支条数；让同 warp 线程走同一支。
- occupancy = 木桶(寄存器/shared/线程上限取最小)；够用即可，不是越高越好。

下一站：**模块 02 · 内存层级与合并访问** —— warp 不只锁步执行，还锁步*访存*。